you need to install some Python libraries that will help you with cryptographic functions and data encoding. The key libraries used are:

ecdsa: Used for elliptic curve cryptography to generate private and public keys.

hashlib: Provides hashing algorithms such as SHA-256, crucial for creating addresses.

base58: Encoding used for generating the final wallet address format.

In [1]:
!pip install ecdsa base58 pycryptodome

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- -------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 1: Generate a Private Key
With every application in Python, the frist thing you need to do for your wallet application is to generate a private key, a 256-bit number that will be used to sign transactions and access funds. In Python, we can use the os.urandom() function to create this random 32-byte private key.

In [2]:
import os

private_key = os.urandom(32)
print(f"Private Key: {private_key.hex()}")

Private Key: 78b0b604e2b6b1fabcc91a5ed0b4095d39458fb2473cadd6e0c28eb04c402364


# Step 2: Generate the Public Key
Once you’ve generated the private key, you can derive the public key. The public key is created using elliptic curve cryptography, specifically the ecdsa (Elliptic Curve Digital Signature Algorithm) library in Python.

We use the SECP256k1 curve, which is widely used in Bitcoin and other cryptocurrencies:

In [3]:
import ecdsa

sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
vk = sk.get_verifying_key()
public_key = b"\x04" + vk.to_string()  # Adding 0x04 byte to signify uncompressed public key
print(f"Public Key: {public_key.hex()}")

Public Key: 04c41f700e51aa64deb319a70ab032b61126267fc0864b53a38505791e620571205e302a4d7a3b9504e4115b5bedffb5b4c0fc0bebb087d31e3d864d3d5a997d8e


# Step 3: Generate the Wallet Address
Now that you’ve acquired a public key, you can generate the wallet address. To do this, you’ll apply several hashing algorithms to the public key and format it in a way that can be used as a wallet address.

The process involves hashing the public key with SHA-256, then applying RIPEMD-160 for further compression, and finally encoding the result with Base58.

In [14]:
import hashlib
import base58
from Cryptodome.Hash import RIPEMD160

# Step 1: SHA-256 Hash of Public Key
sha256_hash = hashlib.sha256(public_key).digest()

# Step 2: RIPEMD-160 Hash of SHA-256
# Use Cryptodome.Hash.RIPEMD160 for ripemd160 hashing
ripemd160_hasher = RIPEMD160.new()
ripemd160_hasher.update(sha256_hash)
ripemd160 = ripemd160_hasher.digest()

# Step 3: Adding Network Byte (0x00 for Bitcoin)
network_byte = b'\x00' + ripemd160

# Step 4: Creating a Checksum by Double Hashing
checksum = hashlib.sha256(hashlib.sha256(network_byte).digest()).digest()[:4]

# Step 5: Adding Checksum to the Versioned RIPEMD-160 Hash
address = base58.b58encode(network_byte + checksum)
print(f"Wallet Address: {address.decode()}")

Wallet Address: 1MtoAfPkFZcdsxWXPu9pLnTLcrCRuK1Bwp


In [15]:
%%writefile crypto_wallet.py
import os
import ecdsa
import hashlib
import base58
from Cryptodome.Hash import RIPEMD160

# Generate a random 32-byte private key
private_key = os.urandom(32)
print(f"Private Key: {private_key.hex()}")

# Derive the public key
sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
vk = sk.get_verifying_key()
public_key = b"\x04" + vk.to_string()  # Adding 0x04 byte for uncompressed public key
print(f"Public Key: {public_key.hex()}")

# Generate the Bitcoin wallet address
# Step 1: SHA-256 Hash of Public Key
sha256_hash = hashlib.sha256(public_key).digest()

# Step 2: RIPEMD-160 Hash of SHA-256
ripemd160_hasher = RIPEMD160.new()
ripemd160_hasher.update(sha256_hash)
ripemd160 = ripemd160_hasher.digest()

# Step 3: Adding Network Byte (0x00 for Bitcoin)
network_byte = b'\x00' + ripemd160

# Step 4: Creating a Checksum by Double Hashing
checksum = hashlib.sha256(hashlib.sha256(network_byte).digest()).digest()[:4]

# Step 5: Adding Checksum to the Versioned RIPEMD-160 Hash
address = base58.b58encode(network_byte + checksum)
print(f"Wallet Address: {address.decode()}")

Writing crypto_wallet.py


In [16]:
!python crypto_wallet.py

Private Key: 62abb3f043010dfd1f7954f393cb52e9864766072d7bdef685013cb42edfdc10
Public Key: 04f87a50e9f7cb49290301f5780bd83133d90ddd6df461d703151c58ef89e34ff9dd7aa4fdc66e964535f147b50c1eacb4138a990eb110581489674299ec37072e
Wallet Address: 15tAWUYExxg4unoZvaHU2R7YL9itthhiA5


Here's the code to create the `crypto_wallet.py` file. This script will generate a new private key, public key, and Bitcoin wallet address each time it's run.